# Librerías 

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import argparse
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from copy import deepcopy
from pathlib import Path
import joblib
from sklearn.metrics import make_scorer, roc_auc_score


import os
import joblib
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt
from scipy.stats import kruskal
from statsmodels.stats.multitest import multipletests
from shap.maskers import Independent


In [2]:
import matplotlib as mpl

import os
import argparse
import numpy as np
import pandas as pd
import re
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

import matplotlib as mpl
mpl.use('Agg')
import scienceplots
plt.style.use(['science', 'grid'])
mpl.rcParams["text.usetex"] = False
dpi = 300

# Librerías para interpretabilidad
import shap
from lime.lime_tabular import LimeTabularExplainer

from copy import deepcopy
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold

# Importación de clasificadores
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

# Herramientas para evaluación y calibración
from sklearn.calibration import CalibratedClassifierCV, CalibrationDisplay
from sklearn.metrics import brier_score_loss
from sklearn.metrics import (
    roc_auc_score, matthews_corrcoef, cohen_kappa_score, f1_score,
    accuracy_score, recall_score, precision_score, balanced_accuracy_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

# Optimización bayesiana de hiperparámetros
from skopt import BayesSearchCV
from skopt.space import Real, Integer, Categorical

import joblib

# Para análisis estadístico de valores SHAP
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
from sklearn.metrics import make_scorer, roc_auc_score
from scipy.stats import kruskal

# FUNCIONES

## SHAP

In [3]:
###########################
#         SHAP            #
###########################

def perform_shap_analysis(X_data, y_data, model_clf, preprocessor, shap_dir, report_path, dataset_name="conjunto"):
    """
    Realiza análisis SHAP sobre un conjunto de datos.
    
    Args:
        X_data: Datos de características sin procesar
        y_data: Etiquetas
        model_clf: Clasificador final
        preprocessor: Pipeline de preprocesamiento
        shap_dir: Directorio donde guardar los resultados
        dataset_name: Nombre del conjunto de datos (para etiquetar)
    """
    print(f"\nRealizando análisis SHAP para {dataset_name}...")
    try:
        print(X_data.shape)
        # Aplicar StandardScaler conservando nombres de columnas
        scaler = preprocessor.steps[0][1]
        X_scaled = pd.DataFrame(scaler.transform(X_data),
                            index=X_data.index,
                            columns=X_data.columns)
        # Aplicar VarianceThreshold y recuperar columnas seleccionadas
        vt = preprocessor.steps[1][1]
        mask = vt.get_support()
        selected_features = X_data.columns[mask]
        X_transformed_array = vt.transform(X_scaled.values)
        X_transformed = pd.DataFrame(X_transformed_array,
                                    index=X_data.index,
                                    columns=selected_features)
        # # Seleccionar el explainer adecuado según el tipo de modelo
        # if isinstance(model_clf, (RandomForestClassifier, GradientBoostingClassifier)):
        #     # Para modelos basados en árboles
        #     explainer = shap.TreeExplainer(model_clf)
        # elif isinstance(model_clf, LogisticRegression):
        #     # Para modelos lineales
        #     try:
        #         explainer = shap.LinearExplainer(model_clf, X_transformed)
        #     except Exception:
        #         # Si falla, usar KernelExplainer como alternativa
        #         background = shap.kmeans(X_transformed, 50)
        #         explainer = shap.KernelExplainer(model_clf.predict_proba, background)
        # else:
        #     # Para otros modelos (SVM, KNN, NaiveBayes)
        #     background = shap.kmeans(X_transformed, 50) # Resumen del dataset para acelerar
        #     explainer = shap.KernelExplainer(model_clf.predict_proba, background)
        
        #prueba
        print(" - Preprocesando datos...")
        classes = np.unique(y_data)
        class_names = {i: f"Clase {c}" for i,c in enumerate(classes)}
        print(f" - Clases detectadas: {class_names}")
        import os, joblib

        shap_cache = os.path.join(shap_dir, "shap_background_cache22.pkl")

        if os.path.exists(shap_cache):
            # load previously‐computed explainer result
            shap_result = joblib.load(shap_cache)
            X_slice = X_transformed #shap_result.data
        else:
            # pick a small background to speed things up
            background = X_transformed.sample(min(200, len(X_transformed)), random_state=42)

            # build the explainer on predict_proba
            explainer = shap.Explainer(
                model_clf.predict_proba,      
                masker=Independent(background),
                feature_names=selected_features,
                output_names=class_names       
            )

            # compute SHAP values (on a slice or on the whole)
            X_slice = X_transformed 
            shap_result = explainer(X_slice)    

            # cache it
            joblib.dump(shap_result, shap_cache)
        # now you can immediately extract and plot:
        all_vals = shap_result.values
        print(f" - Valores SHAP calculados para {dataset_name}.")
        # Verificar si es un problema de clasificación multiclase
        if len(all_vals.shape) == 2:
            # Caso binario o regresión
            shap_values_class = [all_vals]
        elif len(all_vals.shape) == 3:
            # Caso multiclase
            # all_vals tiene forma (n_samples, n_features, n_classes)
            print(f" - Detectado problema de clasificación multiclase con {all_vals.shape[2]} clases.")
            # Extraer valores SHAP por clase
            if all_vals.shape[0] != X_slice.shape[0]:
                raise ValueError("Los valores SHAP no coinciden con el número de muestras en X_slice.")
            if all_vals.shape[1] != X_slice.shape[1]:
                raise ValueError("Los valores SHAP no coinciden con el número de características en X_slice.")
        n_classes = all_vals.shape[2]
        shap_values_class = [all_vals[:, :, i] for i in range(n_classes)]
        
        shap.summary_plot(
            shap_values_class,
            X_slice,           # or X_transformed for full
            feature_names=selected_features,
            class_names=class_names,
            class_inds="original",
            show=False
        )
        plt.savefig(os.path.join(shap_dir, "shap_summary_multiclass2.png"),
                    bbox_inches="tight", dpi=300)
        print(f" - Gráfico de resumen SHAP guardado en {shap_dir}/shap_summary_multiclass2.png")
        plt.close()
        
        # --------------------------------------------------------------
        # PARTE 1: TEST ESTADÍSTICO entre valores SHAP y la clase
        # --------------------------------------------------------------
        print(f" - Realizando test estadístico (Kruskal-Wallis) para {dataset_name} con corrección Holm...")

        # Detectar clases únicas
        unique_classes = np.unique(y_data)
        # Lista para guardar resultados por clase
        shap_stats_results = []

        # Procesar una matriz SHAP por cada clase
        for class_idx, class_shap_values in enumerate(shap_values_class):
            print(f"  > Procesando SHAP para la clase {class_idx}...")

            shap_matrix = pd.DataFrame(
                class_shap_values,
                index=X_transformed.index,
                columns=X_transformed.columns
            )

            features_test = []
            pvalues_raw = []

            for feat in shap_matrix.columns:
                # Agrupar valores SHAP de esta feature por clase
                grouped_values = [shap_matrix.loc[y_data == c, feat] for c in unique_classes]
                try:
                    stat, pval = kruskal(*grouped_values)
                except ValueError:
                    pval = 1.0  # fallback in case one class has no samples
                features_test.append(feat)
                pvalues_raw.append(pval)

            # Corrección Holm por comparaciones múltiples
            alpha = 0.05
            reject, pvals_corr, _, _ = multipletests(pvalues_raw, alpha=alpha, method='holm')

            # Preparar el reporte
            lines_output = []
            lines_output.append("=================================")
            lines_output.append(f"Kruskal-Wallis por feature - SHAP clase {class_idx}")
            lines_output.append(f"alpha = {alpha}")
            lines_output.append(f"Features totales: {len(features_test)}") 
            lines_output.append("=================================\n")
            lines_output.append(f"Resultados por feature (p-valor crudo y corregido):")

            significant_feats = []

            for feat, pval_raw, pval_corr, rej_bool in zip(features_test, pvalues_raw, pvals_corr, reject):
                if rej_bool:
                    result_str = "=> DIFERENCIA SIGNIFICATIVA"
                    significant_feats.append((feat, pval_raw, pval_corr))
                else:
                    result_str = "=> sin diferencia significativa"
                lines_output.append(
                    f"    {feat}: p-valor crudo={pval_raw:.4e}, p-valor corregido={pval_corr:.4e} {result_str}"
                )

            lines_output.append("")
            lines_output.append(f" Total con diferencia significativa: {len(significant_feats)}")

            shap_stats_results.append((class_idx, lines_output))

            # Guardar resultado para esta clase
            test_txt_path = os.path.join(shap_dir, f"shap_statistical_test_class_{class_idx}.txt")
            with open(test_txt_path, "w", encoding="utf-8") as f_out:
                for line in lines_output:
                    f_out.write(line + "\n")

            print(f"    --> Guardado: {test_txt_path}")

    
        # --------------------------------------------------------------
        # PARTE 2: HEATMAP
        # --------------------------------------------------------------
        # print(f" - Generando Heatmap con muestras ordenadas por clase para {dataset_name}...")
        # # Obtener índices ordenados por clase
        # class_labels = np.unique(y_data)
        # idx_order = np.concatenate([np.where(y_data == c)[0] for c in class_labels])
        # class_positions = np.cumsum([len(np.where(y_data == c)[0]) for c in class_labels])
        # for i, class_shap in enumerate(shap_values_class):
        #     print(f" - Generando heatmap para Clase {i}...")
        #     heatmap_path = os.path.join(shap_dir, f"shap_heatmap_class_{i}.png")
            
        #     # shap.plots.heatmap(
        #     #     class_shap,
        #     #     show=False,
        #     #     instance_order=idx_order
        #     # )
        #     shap_explanation = shap.Explanation(
        #         values=class_shap,
        #         data=X_transformed.values,  # Los datos originales como numpy array
        #         feature_names=list(X_transformed.columns)
        #     )
            
        #     shap.plots.heatmap(
        #         shap_explanation,
        #         instance_order=idx_order,
        #         show=False
        #     )
        #     fig = plt.gcf()
        #     ax = plt.gca()
            
        #     # Dibujar líneas divisorias entre clases
        #     for split in class_positions[:-1]:
        #         ax.axvline(split - 0.5, color='black', linewidth=1, zorder=10)

        #     # Etiquetas de clase
        #     n_total = len(idx_order)
        #     prev = 0
        #     for c, split in zip(class_labels, class_positions):
        #         midpoint = (prev + split) / 2 / n_total
        #         ax.text(midpoint, 1.01, f'Clase {c}', ha='center', va='bottom', transform=ax.transAxes)
        #         prev = split

        #     fig.set_size_inches(10, 6)
        #     plt.tight_layout()
        #     plt.savefig(heatmap_path, dpi=300, bbox_inches='tight')
        #     plt.close()
        #     print(f"  --> Heatmap guardado: {heatmap_path}")

        # --------------------------------------------------------------

        # --------------------------------------------------------------
        # PARTE 2: HEATMAP - UNA SOLA IMAGEN CON TODAS LAS CLASES
        # --------------------------------------------------------------
        print(f" - Generando Heatmap con muestras ordenadas por clase para {dataset_name}...")
        
        # Obtener índices ordenados por clase
        class_labels = np.unique(y_data)
        idx_order = np.concatenate([np.where(y_data == c)[0] for c in class_labels])
        class_positions = np.cumsum([len(np.where(y_data == c)[0]) for c in class_labels])
        
        # Usar valores SHAP promediados o de la primera clase
        representative_shap = shap_values_class[0]  # O np.mean(shap_values_class, axis=0)
        
        heatmap_path = os.path.join(shap_dir, f"shap_heatmap_all_classes.png")
        
        shap_explanation = shap.Explanation(
            values=representative_shap,
            data=X_transformed.values,
            feature_names=list(X_transformed.columns)
        )
        
        shap.plots.heatmap(
            shap_explanation,
            instance_order=idx_order,
            show=False
        )
        fig = plt.gcf()
        ax = plt.gca()
        
        # Dibujar líneas divisorias entre clases
        for split in class_positions[:-1]:
            ax.axvline(split - 0.5, color='black', linewidth=2, zorder=10)

        # Etiquetas de clase
        n_total = len(idx_order)
        prev = 0
        for c, split in zip(class_labels, class_positions):
            midpoint = (prev + split) / 2 / n_total
            ax.text(midpoint, 1.01, f'Clase {c}', ha='center', va='bottom', 
                   transform=ax.transAxes, fontweight='bold')
            prev = split

        fig.set_size_inches(12, 8)
        plt.tight_layout()
        plt.savefig(heatmap_path, dpi=300, bbox_inches='tight')
        plt.close()
        print(f"  --> Heatmap guardado: {heatmap_path}")

        # --------------
        # Beeswarm plot 
        # --------------
        dpi = 300
        for i, class_shap in enumerate(shap_values_class):
            shap_fig_path = os.path.join(shap_dir, f"shap_beeswarm_class_{i}.png")
            shap_explanation = shap.Explanation(
                values=class_shap,
                data=X_transformed.values,
                feature_names=list(X_transformed.columns)
            )
            
            shap.plots.beeswarm(shap_explanation, max_display=16, show=False)
         
            # shap.plots.beeswarm(class_shap, max_display=16, show=False)
            fig = plt.gcf()
            fig.set_size_inches(14, 8)
            plt.subplots_adjust(left=0.4, right=0.95)
            plt.tight_layout()
            plt.savefig(shap_fig_path, dpi=dpi, bbox_inches='tight')
            plt.close()
            print(f"  --> Beeswarm plot guardado: {shap_fig_path}")

    
        # --------------------------------------------------------------
        # Scatter plots de las top features
        # --------------------------------------------------------------
        # Crear directorio para gráficos individuales
        # top_features_by_class = []
        # for i, class_shap in enumerate(shap_values_class):
        #     mean_abs_shap = np.abs(class_shap).mean(axis=0)
        #     top_idx = np.argsort(mean_abs_shap)[-15:]
        #     top_idx = top_idx[np.argsort(mean_abs_shap[top_idx])[::-1]]
        #     top_features_shap = X_transformed.columns[top_idx]
        #     top_features_by_class.append(top_features_shap)
            
        #     scatter_dir_class = os.path.join(shap_dir, f"scatter_plots_class_{i}")
        #     os.makedirs(scatter_dir_class, exist_ok=True)

        #     for j, feature in enumerate(top_features_shap, start=1):
        #         scatter_fig_path = os.path.join(scatter_dir_class, f"{j:02d}_{feature}.png")
        #         # feature_idx = X_transformed.columns.get_loc(feature)
        #         feature_idx = list(X_transformed.columns).index(feature)

        #         feature_shap_vals = class_shap[:, feature_idx]
        #         feature_data_vals = X_transformed.iloc[:, feature_idx].values
                

        #         shap_explanation_scatter = shap.Explanation(
        #                             values=feature_shap_vals,
        #                             data=feature_data_vals,
        #                             feature_names=[feature]
        #                         )

        #         # shap.plots.scatter(class_shap[:, feature_idx], color=class_shap, show=False)
        #         shap.plots.scatter(
        #             shap_explanation_scatter, 
        #             color=feature_shap_vals,  # Para el color
        #             show=False
        #         )
        #         fig = plt.gcf()
        #         fig.set_size_inches(10, 6)
        #         plt.tight_layout()
        #         plt.savefig(scatter_fig_path, dpi=dpi, bbox_inches='tight')
        #         plt.close()
            
        #     print(f"  --> Scatter plots guardados para clase {i} en: {scatter_dir_class}")


        return True, selected_features, shap_values_class
    
    except Exception as e:
        with open(report_path, "a", encoding="utf-8") as f_out:
            f_out.write(f"=== SHAP Analysis ({dataset_name}) ===\n")
            f_out.write(" No se pudo generar SHAP (modelo no soportado o error):\n")
            f_out.write(f"  {repr(e)}\n\n")
        print(f"Error en SHAP analysis para {dataset_name}:", e)
        return False, None, None, None

## LIME

In [4]:
###########################
#         LIME            #
###########################

def extraer_nombre(feat_str):
    """
    Extrae el nombre de característica original de la representación de LIME.
    LIME puede añadir información adicional como rangos o categorías.
    """
    tokens = re.findall(r'[A-Za-z0-9_\.\-]+', feat_str)
    valid_tokens = [t for t in tokens if re.search('[A-Za-z]', t)]
    if not valid_tokens:
        return feat_str.strip()
    return max(valid_tokens, key=len)


def explain_lime_instance(
    X_data, 
    index, 
    y_true, 
    y_pred, 
    model_clf, 
    explainer, 
    lime_dir, 
    instance_label="instancia"
):
    """
    Genera y guarda explicación LIME para una instancia específica.
    
    Args:
        X_data: Datos preprocesados
        index: Índice de la instancia a explicar
        y_true: Etiquetas reales
        y_pred: Predicciones del modelo
        model_clf: Clasificador final
        explainer: Explainer LIME configurado
        lime_dir: Directorio para guardar resultados
        instance_label: Etiqueta para identificar la instancia
    """
    
    # Generar explicación LIME
    exp = explainer.explain_instance(
        data_row=X_data[index],
        predict_fn=model_clf.predict_proba,
        num_features=10 # Número de características a mostrar
    )
    
    # Rutas para guardar resultados
    explanation_txt_path = os.path.join(lime_dir, f"lime_explanation_{instance_label}_{index}.txt")
    fig_path = os.path.join(lime_dir, f"lime_explanation_{instance_label}_{index}.png")
    
    # Guardar explicación como texto
    with open(explanation_txt_path, "w", encoding="utf-8") as f:
        f.write(f"=== LIME Explanation para {instance_label} (índice: {index}) ===\n\n")
        f.write(f"Clase real: {y_true[index]}\n")
        f.write(f"Predicción del modelo: {y_pred[index]}\n")
        f.write(f"Probabilidades: {model_clf.predict_proba([X_data[index]])}\n\n")
        f.write("Importancia local de las features:\n")
        for feat_info in exp.as_list():
            f.write("  {}: {:.4f}\n".format(feat_info[0], feat_info[1]))
    
    # Generar y guardar visualización
    with plt.style.context("default"):
        lime_fig = exp.as_pyplot_figure()
        ax = plt.gca()
        
        pos_color = "#0072B2"   
        neg_color = "#E69F00"   
    
        for rect in ax.patches:
            if rect.get_facecolor() == (0.0, 1.0, 0.0, 1.0): 
                rect.set_facecolor(pos_color)
            elif rect.get_facecolor() == (1.0, 0.0, 0.0, 1.0):
                rect.set_facecolor(neg_color)
        
        # plt.title(f"LIME Explanation - {instance_label} (index={index})")
        plt.savefig(fig_path, dpi=dpi, bbox_inches='tight')
        plt.close(lime_fig)
    
    print(f"  -> LIME para {instance_label} (índice {index}) guardado en:\n"
          f"     {explanation_txt_path}\n"
          f"     {fig_path}")

def generate_lime_explanations_for_multiclass(
    X_test_lime,
    y_test,
    model_clf, 
    explainer,
    lime_dir
):
    """
    Genera explicaciones LIME para una instancia correctamente clasificada y
    una mal clasificada de cada clase (si existen).
    """
    y_pred = model_clf.predict(X_test_lime)
    unique_classes = np.unique(y_test)

    for class_label in unique_classes:
        # Casos correctamente clasificados (verdaderos positivos para la clase)
        correct_indices = np.where((y_test == class_label) & (y_pred == class_label))[0]
        if len(correct_indices) > 0:
            explain_lime_instance(
                X_data=X_test_lime,
                index=correct_indices[0],
                y_true=y_test,
                y_pred=y_pred,
                model_clf=model_clf,
                explainer=explainer,
                lime_dir=lime_dir,
                instance_label=f"Clase{class_label}_Correcto"
            )

        # Casos mal clasificados (falsos negativos para la clase)
        incorrect_indices = np.where((y_test == class_label) & (y_pred != class_label))[0]
        if len(incorrect_indices) > 0:
            explain_lime_instance(
                X_data=X_test_lime,
                index=incorrect_indices[0],
                y_true=y_test,
                y_pred=y_pred,
                model_clf=model_clf,
                explainer=explainer,
                lime_dir=lime_dir,
                instance_label=f"Clase{class_label}_Error"
            )

def perform_lime_analysis(X_data, y_data, model_clf, preprocessor, lime_dir, selected_features, report_path, shap_top_features=None, dataset_name="conjunto"):
    """
    Realiza análisis LIME sobre un conjunto de datos.
    
    Args:
        X_data: Datos de características sin procesar
        y_data: Etiquetas 
        model_clf: Clasificador final
        preprocessor: Pipeline de preprocesamiento
        lime_dir: Directorio donde guardar los resultados
        selected_features: Lista de características seleccionadas
        shap_top_features: Top features según SHAP (opcional, para ordenamiento consistente)
        dataset_name: Nombre del conjunto de datos (para etiquetar)
    """
    print(f"\nRealizando análisis LIME para {dataset_name}...")
    try:
        # Preprocesar datos
        X_lime = preprocessor.transform(X_data)
        
        # Configurar explainer LIME
        unique_classes = np.unique(y_data)
        explainer_lime = LimeTabularExplainer(
            training_data=X_lime,
            feature_names=selected_features,
            class_names=[str(c) for c in unique_classes],
            discretize_continuous=True,
            random_state=42
        )        
        # Recopilar resultados de LIME para todas las instancias
        resultados = []
        print(len(X_lime))
        # Iterar en múltiples instancias para análisis global
        for i in range(len(X_lime)):
            exp = explainer_lime.explain_instance(
                data_row=X_lime[i],
                predict_fn=model_clf.predict_proba,
                num_features=10,  # Número de características a mostrar
                labels=unique_classes
            )
            pred_class = model_clf.predict([X_lime[i]])[0]
            lime_list = exp.as_list(label=pred_class)

            
            # Procesar cada característica y su importancia
            for (feat_str, peso) in lime_list:
                feature_name = extraer_nombre(feat_str) 
                col_idx = selected_features.get_loc(feature_name)
                valor_feature = X_lime[i, col_idx]
                
                # Almacenar resultados
                resultados.append({
                    'instancia': i,
                    'feature': feature_name,
                    'peso': peso,
                    'valor_feature': valor_feature
                })
        
        df_lime = pd.DataFrame(resultados)
        df_lime['abs_peso'] = df_lime['peso'].abs()
        
        # Seleccionar las 15 características con mayor peso absoluto promedio
        top_features = (
            df_lime.groupby('feature')['abs_peso']
            .mean()
            .sort_values(ascending=False)
            .head(15)
            .index
            .tolist()
        )
        
        # Filtrar DataFrame para mostrar solo las características principales
        df_lime_top15 = df_lime[df_lime['feature'].isin(top_features)].copy()
        
        # Normalización min-max para cada característica
        df_lime_top15['valor_min'] = df_lime_top15.groupby('feature')['valor_feature'].transform('min')
        df_lime_top15['valor_max'] = df_lime_top15.groupby('feature')['valor_feature'].transform('max')
        
        # Normalización min-max (escala 0-1)
        df_lime_top15['valor_feature_norm'] = (
            (df_lime_top15['valor_feature'] - df_lime_top15['valor_min'])
            / (df_lime_top15['valor_max'] - df_lime_top15['valor_min'])
        )
        
        # Normalización logarítmica
        # Asegurar valores positivos
        df_lime_top15['valor_feature_shifted'] = df_lime_top15.groupby('feature')['valor_feature'].transform(
            lambda x: x - x.min() + 1e-10 if x.min() <= 0 else x
        )
        # Aplicar transformación logarítmica
        df_lime_top15['valor_feature_log'] = np.log1p(df_lime_top15['valor_feature_shifted'])
        # Normalizar los valores logarítmicos
        df_lime_top15['valor_feature_log_min'] = df_lime_top15.groupby('feature')['valor_feature_log'].transform('min')
        df_lime_top15['valor_feature_log_max'] = df_lime_top15.groupby('feature')['valor_feature_log'].transform('max')
        df_lime_top15['valor_feature_log_norm'] = (
            (df_lime_top15['valor_feature_log'] - df_lime_top15['valor_feature_log_min'])
            / (df_lime_top15['valor_feature_log_max'] - df_lime_top15['valor_feature_log_min'])
        )
        
        # Ordenar características para visualización
        # Usar mismo orden que SHAP cuando sea posible
        lime_features = df_lime_top15['feature'].unique().tolist()
        
        # Crear orden final (primero SHAP, luego el resto de LIME)
        if shap_top_features is not None:
            shap_order = list(shap_top_features)
            final_order = [feat for feat in shap_order if feat in lime_features] + \
                         [feat for feat in lime_features if feat not in shap_order]
        else:
            final_order = lime_features
                      
        # Definir colores comunes para ambos gráficos
        colors = [
            (0.0,  "#008afb"),
            (0.2, "#008afb"),  
            (0.7, "#ff0052"),  
            (1.0,  "#ff0052")  
        ]
        
        # Crear colormap personalizado
        the_cmap = LinearSegmentedColormap.from_list("my_cmap", colors)
        
        # --- Gráfico 1: Visualización min-max ---
        fig, ax = plt.subplots(figsize=(10,8))
        sns.stripplot(
            data=df_lime_top15,
            x='peso',                  # Peso LIME en eje X
            y='feature',               # Características en eje Y
            hue='valor_feature_norm',  # Color según valor normalizado
            palette=the_cmap,          # Paleta personalizada
            hue_norm=(0, 1),           # Rango de normalización
            orient='h',                # Horizontal
            size=5,                    # Tamaño de puntos
            dodge=False,               # Sin separación
            legend=False,              # Sin leyenda independiente
            order=final_order,         # Orden personalizado de características
            ax=ax
        )
        
        ax.axvline(0, color='black', linestyle='--')
        # ax.set_title(f"Distribución de pesos LIME - Top 15 features ({dataset_name}, Normalización Min-Max)")
        
        norm = mpl.colors.Normalize(vmin=0, vmax=1)
        sm = mpl.cm.ScalarMappable(cmap=the_cmap, norm=norm)
        sm.set_array([])
        
        cbar = fig.colorbar(sm, ax=ax)
        cbar.set_label("Valor feature normalizado [0..1]")
        
        plt.tight_layout()
        fig_path = os.path.join(lime_dir, "lime_pseudo_beeswarm.png")
        plt.savefig(fig_path, dpi=dpi, bbox_inches='tight')
        plt.close()
        
        # --- Gráfico 2: Visualización logarítmica ---
        fig, ax = plt.subplots(figsize=(10,8))
        sns.stripplot(
            data=df_lime_top15,
            x='peso',
            y='feature',
            hue='valor_feature_log_norm',   # Valor log-normalizado  
            palette=the_cmap,          
            hue_norm=(0, 1),             
            orient='h',
            size=5,
            dodge=False,
            legend=False,
            order=final_order,
            ax=ax
        )
        
        ax.axvline(0, color='black', linestyle='--')
        # ax.set_title(f"Distribución de pesos LIME - Top 15 features ({dataset_name}, Normalización Logarítmica)")
        
        norm = mpl.colors.Normalize(vmin=0, vmax=1)
        sm = mpl.cm.ScalarMappable(cmap=the_cmap, norm=norm)
        sm.set_array([])
        
        cbar = fig.colorbar(sm, ax=ax)
        cbar.set_label("Valor feature norm. logarítmica [0..1]")
        
        plt.tight_layout()
        fig_path_log = os.path.join(lime_dir, "lime_pseudo_beeswarm_log.png")
        plt.savefig(fig_path_log, dpi=dpi, bbox_inches='tight')
        plt.close()
        
        print(f"  --> Visualización LIME (normalización min-max) guardada en: {fig_path}")
        print(f"  --> Visualización LIME (normalización logarítmica) guardada en: {fig_path_log}")
        
        # --- Explicaciones locales para casos específicos ---
        ind_lime_dir = os.path.join(lime_dir, "individual_analysis")
        os.makedirs(ind_lime_dir, exist_ok=True)
        
        # Generar explicaciones para casos representativos
        generate_lime_explanations_for_multiclass(
            X_test_lime=X_lime,
            y_test=y_data,
            model_clf=model_clf,
            explainer=explainer_lime,
            lime_dir=ind_lime_dir
        )
    
        return True
    
    except Exception as e:
        with open(report_path, "a", encoding="utf-8") as f_out:
            f_out.write(f"\n=== LIME Analysis ({dataset_name}) ===\n")
            f_out.write("No se pudo generar LIME (modelo no soportado o error):\n")
            f_out.write(f"  {repr(e)}\n\n")
        print(f"Error en LIME analysis para {dataset_name}:", e)
        return False


# Paths

In [6]:
# Paths and settings
path_features        = '/mnt/datalake/openmind/MedP-Midas/sgonzalez/radiomics-midas-new/binary/features_t2w_MPfirrmann.csv'
experiment_dir       = '/mnt/datalake/openmind/MedP-Midas/sgonzalez/radiomics-midas-new/data/features_t2w_multiclass'
variables_txt_path   = os.path.join(experiment_dir, 'variables_usadas.txt')
best_model_finetune  = 'GradientBoosting'
n_folds              = 5

# Load data
df = pd.read_csv(path_features)
print("Configuration:")
print(f"  Features CSV:       {path_features}")
print(f"  Experiment dir:     {experiment_dir}")
print(f"  Variables file:     {variables_txt_path}")
print(f"  Model for fine-tune:{best_model_finetune}")
print(f"  CV folds:           {n_folds}")
print(f"\nLoaded DataFrame with shape {df.shape}")


Configuration:
  Features CSV:       /mnt/datalake/openmind/MedP-Midas/sgonzalez/radiomics-midas-new/binary/features_t2w_MPfirrmann.csv
  Experiment dir:     /mnt/datalake/openmind/MedP-Midas/sgonzalez/radiomics-midas-new/data/features_t2w_multiclass
  Variables file:     /mnt/datalake/openmind/MedP-Midas/sgonzalez/radiomics-midas-new/data/features_t2w_multiclass/variables_usadas.txt
  Model for fine-tune:GradientBoosting
  CV folds:           5

Loaded DataFrame with shape (3580, 1450)


In [7]:
# Configuración de rutas y directorios de salida
selected_model = best_model_finetune
base_dir = os.path.dirname(os.path.abspath(variables_txt_path))
output_parent_dir = os.path.join(base_dir, f"best_results")
calibration_dir = os.path.join(output_parent_dir, "calibration")
explicability_dir = os.path.join(output_parent_dir, "explicability")

train_explicability_dir = os.path.join(explicability_dir, "train")
test_explicability_dir = os.path.join(explicability_dir, "test")

# Subdirectorios SHAP y LIME para train
train_shap_dir = os.path.join(train_explicability_dir, "SHAP")
train_lime_dir = os.path.join(train_explicability_dir, "LIME")

# Subdirectorios SHAP y LIME para test
test_shap_dir = os.path.join(test_explicability_dir, "SHAP")
test_lime_dir = os.path.join(test_explicability_dir, "LIME")

# Crear directorios si no existen
os.makedirs(output_parent_dir, exist_ok=True)
os.makedirs(calibration_dir, exist_ok=True)
os.makedirs(explicability_dir, exist_ok=True)
os.makedirs(train_explicability_dir, exist_ok=True)
os.makedirs(test_explicability_dir, exist_ok=True)
os.makedirs(train_shap_dir, exist_ok=True)
os.makedirs(train_lime_dir, exist_ok=True)
os.makedirs(test_shap_dir, exist_ok=True)
os.makedirs(test_lime_dir, exist_ok=True)

print(f"\nCarpeta de salida creada/ubicada en: {os.path.relpath(output_parent_dir)}")


Carpeta de salida creada/ubicada en: ../data/features_t2w_multiclass/best_results


# Data

In [8]:
# ----------------------------------------------------------------------
# 1) CARGAR CSV E IDENTIFICAR X, y, groups
# ----------------------------------------------------------------------
df = pd.read_csv(path_features)

df['patient_id_type'] = df['patient_id'].astype(str)
df = df.set_index('patient_id_type')
print(f"Datos cargados. Dimensiones: {df.shape}")


# Preparar variables para el modelado
y = df['label'].apply(lambda x: x-1).values
groups = df['patient_id'].values
# X = df.drop(columns=['patient_id'])
X = df.drop([ 'patient_id','study_id','label', 'mask_type',
                              'diagnostics_Versions_PyRadiomics', 'diagnostics_Versions_Numpy', 
                              'diagnostics_Versions_SimpleITK', 'diagnostics_Versions_PyWavelet', 
                              'diagnostics_Versions_Python', 'diagnostics_Configuration_Settings', 
                              'diagnostics_Configuration_EnabledImageTypes', 'diagnostics_Image-original_Hash', 
                              'diagnostics_Image-original_Dimensionality', 'diagnostics_Image-original_Spacing', 
                              'diagnostics_Image-original_Size', 'diagnostics_Image-original_Mean', 
                              'diagnostics_Image-original_Minimum', 'diagnostics_Image-original_Maximum', 
                              'diagnostics_Mask-original_Hash', 'diagnostics_Mask-original_Spacing', 
                              'diagnostics_Mask-original_Size', 'diagnostics_Mask-original_BoundingBox', 
                              'diagnostics_Mask-original_VoxelNum', 'diagnostics_Mask-original_VolumeNum', 
                              'diagnostics_Mask-original_CenterOfMassIndex', 'diagnostics_Mask-original_CenterOfMass', 
                              'diagnostics_Image-interpolated_Spacing', 'diagnostics_Image-interpolated_Size', 
                              'diagnostics_Image-interpolated_Mean', 'diagnostics_Image-interpolated_Minimum', 
                              'diagnostics_Image-interpolated_Maximum', 'diagnostics_Mask-interpolated_Spacing', 
                              'diagnostics_Mask-interpolated_Size', 'diagnostics_Mask-interpolated_BoundingBox', 
                              'diagnostics_Mask-interpolated_VoxelNum', 'diagnostics_Mask-interpolated_VolumeNum', 
                              'diagnostics_Mask-interpolated_CenterOfMassIndex', 'diagnostics_Mask-interpolated_CenterOfMass', 
                              'diagnostics_Mask-interpolated_Mean', 'diagnostics_Mask-interpolated_Minimum', 
                              'diagnostics_Mask-interpolated_Maximum'], axis=1)


# ----------------------------------------------------------------------
# 1.1) FILTRAR LAS VARIABLES USADAS (variables_usadas.txt)
# ----------------------------------------------------------------------
print(f"\nFiltrando variables usando el archivo: {variables_txt_path}")
with open(variables_txt_path, "r", encoding="utf-8") as f_vars:
    used_vars = [line.strip() for line in f_vars if line.strip()]
X = X[used_vars]

# ----------------------------------------------------------------------
# 2) SEPARAR HOLD-OUT TEST SET Y CONJUNTO DE ENTRENAMIENTO
# ----------------------------------------------------------------------
gss = GroupShuffleSplit(test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_train_full, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train_full, y_test = y[train_idx], y[test_idx]
groups_train_full = groups[train_idx]

#Split data for small debugging
# max_debug = 200
# n = len(X_train_full)
# if n > max_debug:
#     # pick 200 random integer positions
#     sel = np.random.RandomState(42).choice(n, size=max_debug, replace=False)
#     X_data = X_train_full.iloc[sel]
#     y_data = y_train_full[sel]
#     groups_train_full = groups_train_full[sel]

X_data = X_train_full
y_data = y_train_full


roc_auc_ovr = make_scorer(
    roc_auc_score,
    response_method="predict_proba",
    multi_class="ovr",
    average="macro"
)

score_group = {
    "roc_auc_ovr": roc_auc_ovr,
    "f1":            "f1_macro",
    "balanced_accuracy": "balanced_accuracy"
}
score_refit_str = "roc_auc_ovr"
random_state_value = 42

Datos cargados. Dimensiones: (3580, 1450)

Filtrando variables usando el archivo: /mnt/datalake/openmind/MedP-Midas/sgonzalez/radiomics-midas-new/data/features_t2w_multiclass/variables_usadas.txt


In [9]:
number_folds = 5
selected_model = "GradientBoosting"  # Modelo seleccionado para fine-tuning
# --- Configuración específica para cada tipo de modelo ---
if selected_model == 'SVM':
    # Pipeline para SVM: Escalado → Filtro varianza → SVM
    pipe = make_pipeline(StandardScaler(),
                        VarianceThreshold(),
                        SVC(random_state=random_state_value, probability=True))
    # Espacio de búsqueda para hiperparámetros
    param_grid = {
        'svc__C': Real(1e-4, 1e3, prior='log-uniform'),            # Regularización
        'svc__kernel': Categorical(['linear', 'rbf', 'poly']),     # Tipo de kernel
        'svc__gamma': Real(1e-4, 1e3, prior='log-uniform'),        # Parámetro gamma
        'svc__coef0': Real(0, 1)                                   # Término independiente (para poly)
    }
    
elif selected_model == 'LogisticRegression':
    # Pipeline para Regresión Logística
    pipe = make_pipeline(StandardScaler(),
                        VarianceThreshold(),
                        LogisticRegression(
                            class_weight='balanced', 
                            random_state=random_state_value,
                            solver='saga',  
                            max_iter=10000
                        ))
    # Espacio de búsqueda
    param_grid = {
        'logisticregression__C': Real(1e-4, 1e3, prior='log-uniform'),  # Regularización
        'logisticregression__penalty': Categorical(['l1', 'l2', 'elasticnet']),  # Tipo de regularización
        'logisticregression__l1_ratio': Real(0.1, 0.9)                  # Ratio L1/L2 para elasticnet
    }
    
elif selected_model == 'RandomForest':
    # Pipeline para Random Forest
    pipe = make_pipeline(StandardScaler(),
                        VarianceThreshold(),
                        RandomForestClassifier(n_jobs=-1, 
                                                class_weight="balanced_subsample", 
                                                random_state=random_state_value))
    # Espacio de búsqueda
    param_grid = {
        'randomforestclassifier__n_estimators': Integer(50, 1024),       # Número de árboles
        'randomforestclassifier__max_depth': Integer(1, 10),             # Profundidad máxima
        'randomforestclassifier__max_features': Categorical(['sqrt', 'log2', None]),  # Features por árbol
        'randomforestclassifier__min_samples_split': Integer(2, 20)      # Min muestras para dividir nodo
    }
    
elif selected_model == 'NaiveBayes':
    # Pipeline para Naive Bayes
    pipe = make_pipeline(StandardScaler(),
                        VarianceThreshold(),
                        GaussianNB())
    param_grid = {}  # Naive Bayes no tiene hiperparámetros a optimizar
    
elif selected_model == 'KNN':
    # Pipeline para K-Nearest Neighbors
    pipe = make_pipeline(StandardScaler(),
                        VarianceThreshold(),
                        KNeighborsClassifier(n_jobs=-1))
    # Espacio de búsqueda
    param_grid = {
        'kneighborsclassifier__n_neighbors': Integer(2, 8),            # Número de vecinos
        'kneighborsclassifier__weights': Categorical(['uniform', 'distance'])  # Ponderación
    }
    
elif selected_model == 'GradientBoosting':
    # Pipeline para Gradient Boosting
    pipe = make_pipeline(StandardScaler(),
                        VarianceThreshold(),
                        GradientBoostingClassifier(random_state=random_state_value))
    # Espacio de búsqueda
    param_grid = {
        'gradientboostingclassifier__n_estimators': Integer(50, 1024),        # Número de árboles
        'gradientboostingclassifier__learning_rate': Real(1e-4, 0.1, prior='log-uniform'),  # Tasa aprendizaje
        'gradientboostingclassifier__max_depth': Integer(1, 10),              # Profundidad máxima
        'gradientboostingclassifier__subsample': Real(0.5, 1.0),              # Fracción muestras por árbol
        'gradientboostingclassifier__max_features': Categorical(['sqrt', 'log2', None])  # Features por árbol
    }
else:
    raise ValueError(f"Modelo '{selected_model}' no reconocido.")



In [ ]:
# ----------------------------------------------------------------------
# 4) AJUSTAR CON BayesSearchCV (OPTIMIZACIÓN BAYESIANA) SOBRE EL CONJUNTO DE ENTRENAMIENTO
# ----------------------------------------------------------------------
cv = StratifiedGroupKFold(n_splits=number_folds, shuffle=True, random_state=random_state_value)
print("\nIniciando optimización bayesiana con BayesSearchCV...")

# Configurar búsqueda bayesiana
search = BayesSearchCV(
    estimator=pipe,
    search_spaces=param_grid,
    scoring=score_group,       # Evaluación con múltiples métricas
    refit=score_refit_str,     # Reentrenar con la mejor configuración según AUC
    cv=cv,                     # Validación cruzada estratificada por grupo
    n_jobs=-1,                 # Usar todos los núcleos disponibles
    random_state=random_state_value
)

search.fit(X_train_full, y_train_full, groups=groups_train_full)
best_estimator = search.best_estimator_
print("\nOptimización completada.")
print(f"  --> Mejores parámetros: {search.best_params_}")


Iniciando optimización bayesiana con BayesSearchCV...


KeyboardInterrupt: 

In [10]:
# # Guardar el mejor modelo
estimator_path = os.path.join(output_parent_dir, "best_estimator.pkl")
joblib.dump(best_estimator, estimator_path)
print(f"  --> Mejor estimador guardado en: {os.path.relpath(estimator_path)}")

search_path = os.path.join(output_parent_dir, "search_results.pkl")
joblib.dump(search, search_path)
print(f"  --> Resultados de búsqueda guardados en: {os.path.relpath(search_path)}")

NameError: name 'best_estimator' is not defined

In [ ]:
output_parent_dir = Path("../data/features_t2w_multiclass/best_results")
estimator_path     = output_parent_dir / "best_estimator.pkl"
search_path = output_parent_dir / "search_results.pkl"

# load the fitted pipeline
best_estimator = joblib.load(estimator_path)
search = joblib.load(search_path)

# Extraer el preprocesador (todos los pasos excepto el clasificador final)
preprocessor = deepcopy(best_estimator)
preprocessor.steps.pop(-1)

# Extraer el clasificador final
model_clf = best_estimator.steps[-1][1]
# pull out only the GB parameters
gb_params = {
    k: v
    for k, v in best_estimator.get_params().items()
    if k.startswith("gradientboostingclassifier__")
}

print("Optimización completada.")
print(" → Mejores parámetros:", gb_params)
print(" → Mejor estimador cargado desde:", estimator_path)

Optimización completada.
 → Mejores parámetros: {'gradientboostingclassifier__ccp_alpha': 0.0, 'gradientboostingclassifier__criterion': 'friedman_mse', 'gradientboostingclassifier__init': None, 'gradientboostingclassifier__learning_rate': 0.0001, 'gradientboostingclassifier__loss': 'log_loss', 'gradientboostingclassifier__max_depth': 5, 'gradientboostingclassifier__max_features': 'sqrt', 'gradientboostingclassifier__max_leaf_nodes': None, 'gradientboostingclassifier__min_impurity_decrease': 0.0, 'gradientboostingclassifier__min_samples_leaf': 1, 'gradientboostingclassifier__min_samples_split': 2, 'gradientboostingclassifier__min_weight_fraction_leaf': 0.0, 'gradientboostingclassifier__n_estimators': 1024, 'gradientboostingclassifier__n_iter_no_change': None, 'gradientboostingclassifier__random_state': 42, 'gradientboostingclassifier__subsample': 0.5, 'gradientboostingclassifier__tol': 0.0001, 'gradientboostingclassifier__validation_fraction': 0.1, 'gradientboostingclassifier__verbose':

In [ ]:
report_path = os.path.join(output_parent_dir, "report.txt")
with open(report_path, "w", encoding="utf-8") as f_out:
    f_out.write(f"=== Fine-tuning del modelo {selected_model} ===\n\n")
    f_out.write(f"Mejores parámetros (según {score_refit_str}): {search.best_params_}\n\n")
    f_out.write("=== Resultados CV (BayesSearch) ===\n")
    idx_best = search.best_index_
    for key in score_group:
        mean_test = search.cv_results_[f'mean_test_{key}'][idx_best]
        std_test  = search.cv_results_[f'std_test_{key}'][idx_best]
        f_out.write(f"  CV {key}: {mean_test:.3f} +/- {std_test:.3f}\n")
    f_out.write("\n")

# SHAP

In [ ]:
report_path = os.path.join(output_parent_dir, "report.txt")

# Realizar análisis SHAP para conjunto de entrenamiento
train_success, selected_features, train_shap_values = perform_shap_analysis(
    X_data=X_train_full,
    y_data=y_train_full,
    model_clf=model_clf,
    preprocessor=preprocessor,
    shap_dir=train_shap_dir,
    report_path=report_path,
    dataset_name="entrenamiento"
)

NameError: name 'output_parent_dir' is not defined

# LIME


In [53]:
train_lime_success = perform_lime_analysis(
    X_data=X_train_full,
    y_data=y_train_full,
    model_clf=model_clf,
    preprocessor=preprocessor,
    lime_dir=train_lime_dir,
    selected_features=selected_features,
    report_path=report_path,
    # shap_top_features=train_top_features,
    dataset_name="entrenamiento"
)


Realizando análisis LIME para entrenamiento...
2862
Error en LIME analysis para entrenamiento: only integers, slices (`:`), ellipsis (`...`), numpy.newaxis (`None`) and integer or boolean arrays are valid indices


# pruebas

In [13]:
scaler = preprocessor.steps[0][1]
X_scaled = pd.DataFrame(
    scaler.transform(X_data),
    index=X_data.index,
    columns=X_data.columns
)

vt = preprocessor.steps[1][1]
mask = vt.get_support()
selected_features = X_data.columns[mask]
X_transformed = pd.DataFrame(
    vt.transform(X_scaled.values),
    index=X_data.index,
    columns=selected_features
)

print("Preprocessing done:", X_transformed.shape)


Preprocessing done: (2862, 239)


In [14]:
# 2) load or compute shap_result
shap_cache = os.path.join(train_shap_dir, "shap_background_cache.pkl")
if os.path.exists(shap_cache):
    print("→ loading cached SHAP result")
    shap_result = joblib.load(shap_cache)
else:
    background = X_transformed.sample(min(200, len(X_transformed)), random_state=42)
    explainer = shap.Explainer(
        model_clf.predict_proba,
        masker=Independent(background),
        feature_names=selected_features,
        output_names=[f"Clase {c}" for c in np.unique(y_data)]
    )
    shap_result = explainer(X_transformed)
    joblib.dump(shap_result, shap_cache)
    print("→ computed & cached SHAP result")

all_vals = shap_result.values                       # (n_samples, n_features, n_classes)
n_classes = all_vals.shape[2]
shap_values_class = [ all_vals[:, :, i] for i in range(n_classes) ]
print(f"SHAP values shape: {all_vals.shape}")
print(f"Shap values per class: {shap_values_class}")
print(n_classes)


→ loading cached SHAP result
SHAP values shape: (200, 239, 5)
Shap values per class: [array([[ 3.42938825e-06, -2.69688070e-05,  2.79323032e-07, ...,
        -3.87174955e-05,  1.47886050e-05, -5.36205735e-07],
       [ 1.30577269e-05, -1.66846132e-05,  1.07603415e-05, ...,
        -3.11214205e-06,  1.24677556e-05, -5.01406115e-06],
       [-2.00095625e-06, -2.21053563e-05, -1.28804926e-06, ...,
        -3.25442127e-06,  2.05204592e-05, -5.64861786e-06],
       ...,
       [-7.39102319e-06, -3.78864957e-05, -4.09201529e-06, ...,
        -1.14050803e-05,  1.96093923e-05, -3.83722087e-06],
       [ 3.55632816e-07, -2.98378496e-05, -1.14581574e-06, ...,
        -1.20073577e-05,  1.47334900e-05, -1.92206804e-06],
       [ 2.72863215e-06, -2.29391794e-05,  3.80160515e-06, ...,
        -2.02245253e-06,  1.39393551e-05, -8.60040694e-06]]), array([[ 3.33071846e-05, -4.14250764e-05, -1.28859349e-04, ...,
         6.11308894e-05, -2.71080259e-04, -2.49867227e-05],
       [ 4.61546175e-05,  1.9368

In [15]:
# 3) summary beeswarm
shap.summary_plot(
    shap_values_class,    # list-of-arrays for each class
    X_transformed,
    feature_names=selected_features,
    class_names=[f"Clase {c}" for c in np.unique(y_data)],
    class_inds="original",
    show=False
)
plt.savefig(os.path.join(train_shap_dir, "shap_summary_multiclass.png"), dpi=300, bbox_inches="tight")
plt.close()
print("Saved summary plot")
print(train_shap_dir)

Saved summary plot
/mnt/datalake/openmind/MedP-Midas/sgonzalez/radiomics-midas-new/data/features_t2w_multiclass/best_results/explicability/train/SHAP


In [42]:
unique_classes = np.unique(y_data)
for class_idx, class_vals in enumerate(shap_values_class):
    df_sh = pd.DataFrame(
        class_vals,
        index=X_transformed.index,
        columns=selected_features
    )
    p_raw, feats = [], []
    for feat in df_sh.columns:
        groups_list = [ df_sh.loc[y_data==c, feat] for c in unique_classes ]
        stat, p = kruskal(*groups_list)
        feats.append(feat)
        p_raw.append(p)
    reject, p_corr, *_ = multipletests(p_raw, alpha=0.05, method='holm')
    # write out only significant ones
    with open(os.path.join(train_shap_dir, f"shap_kw_class{class_idx}.txt"), "w") as f:
        for feat, pr, pc, rej in zip(feats, p_raw, p_corr, reject):
            f.write(f"{feat}: p_raw={pr:.3e}, p_corr={pc:.3e} {'*' if rej else ''}\n")
    print(f"Krustal - Wall results for class {class_idx}")


Krustal - Wall results for class 0
Krustal - Wall results for class 1
Krustal - Wall results for class 2
Krustal - Wall results for class 3
Krustal - Wall results for class 4


In [47]:
idx_order = np.concatenate([np.where(y_data==c)[0] for c in unique_classes])
for i, class_sh in enumerate(shap_values_class):
    # heatmap
    shap.plots.heatmap(
        class_sh, 
        feature_values=X_transformed,        # <-- here!
        instance_order=idx_order,
        show=False
    )
    plt.savefig(os.path.join(train_shap_dir, f"heatmap_class{i}.png"), dpi=300)
    plt.close()

    # beeswarm (this one can take list-of-arrays directly, no change needed)
    shap.plots.beeswarm(class_sh, max_display=16, show=False)
    plt.savefig(os.path.join(train_shap_dir, f"beeswarm_class{i}.png"), dpi=300)
    plt.close()

    print(f"Plots for class {i}")


AttributeError: 'numpy.ndarray' object has no attribute 'values'

In [ ]:
for i, class_sh in enumerate(shap_values_class):
    df_sh = pd.DataFrame(class_sh, columns=selected_features)
    mean_abs = df_sh.abs().mean().sort_values(ascending=False)
    tops = mean_abs.head(15).index
    os.makedirs(os.path.join(train_shap_dir, f"scatter_class{i}"), exist_ok=True)
    for j, feat in enumerate(tops,1):
        shap.plots.scatter(class_sh[:, df_sh.columns.get_loc(feat)],
                           color=class_sh, show=False)
        plt.savefig(os.path.join(train_shap_dir, f"scatter_class{i}/{j:02d}_{feat}.png"), dpi=300)
        plt.close()
    print(f"✔️ Scatter (top15) for class {i}")
